# **Traitement de la collection**

### Recharge de la mémoire du projet

In [1]:
import ee
import sys
import os

sys.path.append(os.path.abspath('../'))
from src.gee_memory import get_togo_aoi, get_s2_composite, PROJECT_ID

ee.Authenticate()
try:
    ee.Initialize(project=PROJECT_ID)
    print(f"✅ Google Earth Engine initialisé avec succès sur le projet : {PROJECT_ID}")
except Exception as e:
    if 'not initialized' in str(e):
        ee.Authenticate(auth_mode="notebook")
        ee.Initialize(project=PROJECT_ID)
    else:
        raise e

# --- RECHARGE DE LA MÉMOIRE ---
togo = get_togo_aoi()
s2 = get_s2_composite(togo, year='2025')
print("✅ Mémoire Earth Engine rechargée avec succès.")

✅ Google Earth Engine initialisé avec succès sur le projet : forestwatch-tg
✅ Mémoire Earth Engine rechargée avec succès.


## Ajout d'informations spectrales : les indices et indices dérivés

In [2]:
# 1. Calcul des indices sur l'image unique s2
ndvi = s2.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndwi = s2.normalizedDifference(['B3', 'B8']).rename('NDWI')
ndbi = s2.normalizedDifference(['B11', 'B8']).rename('NDBI')

image_with_indices = s2.addBands([ndvi, ndwi, ndbi])

# 2. Moyenne et Variance locales (Fenêtre 3x3)
kernel = ee.Kernel.square(radius=1)
mean = image_with_indices.select(['NDVI', 'NDWI', 'B4']).reduceNeighborhood(
    reducer=ee.Reducer.mean(), kernel=kernel
).rename(['NDVI_mean', 'NDWI_mean', 'B4_mean'])

variance = image_with_indices.select(['NDVI', 'NDWI', 'B4']).reduceNeighborhood(
    reducer=ee.Reducer.variance(), kernel=kernel
).rename(['NDVI_var', 'NDWI_var', 'B4_var'])

image_with_stats = image_with_indices.addBands([mean, variance])

# 3. Application du GLCM
selected_bands_glcm = ['NDVI', 'NDWI', 'NDBI', 'B8', 'B12']

def rescale_to_byte(img, band_name, min_val, max_val):
    return img.select(band_name).subtract(min_val).divide(max_val - min_val).clamp(0, 1).multiply(255).toByte().rename(band_name)

rescaled_bands = []
for band in selected_bands_glcm:
    if band in ['NDVI', 'NDWI', 'NDBI']:
        # Indices varient de -1 à 1
        rescaled_bands.append(rescale_to_byte(image_with_stats, band, -1, 1))
    else:
        # Bandes spectrales varient de 0 à 1 max (suite au divide par 10000). On prend 0.5 comme max empirique.
        rescaled_bands.append(rescale_to_byte(image_with_stats, band, 0, 0.5))

rescaled_image = ee.Image(rescaled_bands)
glcm_texture = rescaled_image.glcmTexture()

# J'assemble tout
compositeImage = image_with_stats.addBands(glcm_texture)

# Je filtre les bandes finales qui m'interessent
features_to_keep = ['NDVI', 'NDWI', 'NDBI', 'B4', 'B8', 'B12',
                    'NDVI_mean', 'NDWI_mean', 'B4_mean',
                    'NDVI_var', 'NDWI_var', 'B4_var',
                    'NDVI_contrast', 'NDVI_asm', 'NDVI_diss', 'NDVI_ent',
                    'NDWI_contrast', 'NDWI_asm', 'NDWI_diss', 'NDWI_ent',
                    'NDBI_contrast', 'NDBI_asm', 'NDBI_diss', 'NDBI_ent',
                    'B8_contrast', 'B8_asm', 'B8_diss', 'B8_ent',
                    'B12_contrast', 'B12_asm', 'B12_diss', 'B12_ent']

compositeImage = compositeImage.select(features_to_keep)
print("✅ Image composite finale avec toutes les features prête.")

✅ Image composite finale avec toutes les features prête.


## Échantillonage et ajustement des classes cibles

In [12]:
# 1. Chargement et remappage de Dynamic World 2025
target_band = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1") \
    .filterBounds(togo.geometry()) \
    .filterDate('2025-01-01', '2025-12-31') \
    .select('label') \
    .mode() \
    .clip(togo.geometry())

# Correspondance Dynamic World :
# DW: 1(Trees), 5(Shrub), 2(Grass), 4(Crops), 6(Built), 7(Bare), 0(Water), 3(Flooded), 8(Snow)
# To: 0(Forêt), 1(Buissons),2(Savanes),3(Cultures),4(Urbain),5(Sols nus),6(Eau), -1(Ignorer), -1(Ignorer)
remapped_target_band = target_band.remap(
    [1, 5, 2, 4, 6, 7, 0, 3, 8], 
    [0, 1, 2, 3, 4, 5, 6, -1, -1]
)
remapped_target_band = remapped_target_band.updateMask(remapped_target_band.neq(-1))
combinedImage = compositeImage.addBands(remapped_target_band.rename('landcover'))

# 2. Définition de la zone de couverture
study_area=togo

# 3. Échantillonnage Stratifié : 3000 points par classe = 21000 points
samples = combinedImage.stratifiedSample(
    numPoints=3000,
    classBand='landcover',
    region=study_area,
    scale=10,
    geometries=True, # Conserve les coordonnées
    seed=42
)

# 4. Ajout des colonnes lat/lon au dataset
def add_coordinates(feature):
    coords = feature.geometry().coordinates()
    return feature.set({'longitude': coords.get(0), 'latitude': coords.get(1)})

samples_with_coords = samples.map(add_coordinates)
print("✅ Échantillonnage stratifié terminé (21000 points générés).")

✅ Échantillonnage stratifié terminé (21000 points générés).


## Sauvegarde du dataset final

In [13]:
# Lancement de la tâche d'export vers Google Drive
task = ee.batch.Export.table.toDrive(
    collection=samples_with_coords,
    description='forestwatch_dataset_v2',
    folder='Projet_ForestWatch-TG',
    fileNamePrefix='dataset_2025_togo',   
    fileFormat='CSV'
)

task.start()

import time
print("Tâche envoyée aux serveurs de Google.")
print("Création du CSV en cours...")

while task.active():
    print('Statut:', task.status()['state'])
    time.sleep(15)

print('✅ Terminé :', task.status())

Tâche envoyée aux serveurs de Google.
Création du CSV en cours...
Statut: READY
Statut: RUNNING
Statut: RUNNING
Statut: RUNNING
Statut: RUNNING
Statut: RUNNING
Statut: RUNNING
Statut: RUNNING
Statut: RUNNING
Statut: RUNNING
Statut: RUNNING
Statut: RUNNING
Statut: RUNNING
Statut: RUNNING
Statut: RUNNING
✅ Terminé : {'state': 'COMPLETED', 'description': 'forestwatch_dataset_v2', 'priority': 100, 'creation_timestamp_ms': 1774137454396, 'update_timestamp_ms': 1774137702088, 'start_timestamp_ms': 1774137460513, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['https://drive.google.com/#folders/171lc78rEyvvQIDE_nwK6NdzU3GvvvQt8'], 'attempt': 1, 'batch_eecu_usage_seconds': 255035.59375, 'id': 'YJ5TP7SLMLT5OTDKXNFDZCEK', 'name': 'projects/forestwatch-tg/operations/YJ5TP7SLMLT5OTDKXNFDZCEK'}
